# Quantum Computing + AI for Drone Fleet Assignment

**IEEE-style demo notebook** — *Quantum optimization meets mission planning*

## The problem

Imagine **five autonomous drones** (HAWK-01 … HAWK-05) parked at different GPS coordinates. A new **mission waypoint** arrives: we must **assign exactly one drone** to fly it. For this talk we use a simple, interpretable rule: **pick the drone closest to the target** (great-circle distance on Earth).

That is a tiny discrete optimization — classical code solves it instantly. We still walk through a **QAOA** (Quantum Approximate Optimization Algorithm) formulation because it is the same *pattern* used when the problem grows: **many assets, coupling constraints, and richer objectives** where exhaustive search is no longer feasible.

This notebook tells one story: **classical baseline → quantum circuit → side-by-side numbers**.

### What this section does

We install the Python packages this demo needs, then import them.

- **Qiskit** — build quantum circuits and Hamiltonians.
- **Qiskit Aer** — simulate the circuit on your laptop (no cloud quantum account required).
- **NumPy** — small numerical helpers (arrays, timing arrays if needed).
- **Matplotlib** — plot a bar chart when we compare classical vs quantum runtimes later.

Run this once in a fresh environment; if everything is already installed, the `%pip` line completes quickly.

In [ ]:
%pip install -q qiskit qiskit-aer numpy matplotlib

import math
import time

import matplotlib.pyplot as plt
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator
from qiskit_aer import AerSimulator

### What this section does

Here we solve the mission the **classical** way everyone would write first: loop over the five drones, compute distance to the mission point for each, and take the **argmin** (smallest distance wins).

We wrap that in `time.time()` so we have a **wall-clock baseline** to compare against the simulator-backed quantum pipeline in the next section. For only five drones this will be **microseconds** — that is expected and part of the story.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance between two WGS84 points (km)."""
    r_earth = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlmb / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(max(0.0, 1.0 - a)))
    return r_earth * c


# Fixed toy fleet + mission (reproducible story for the audience)
MISSION_LAT, MISSION_LON = 32.78, -96.80
drones = [
    {"name": "HAWK-01", "lat": 32.72, "lon": -96.85},
    {"name": "HAWK-02", "lat": 32.81, "lon": -96.76},
    {"name": "HAWK-03", "lat": 32.65, "lon": -96.90},
    {"name": "HAWK-04", "lat": 32.79, "lon": -96.95},
    {"name": "HAWK-05", "lat": 32.70, "lon": -96.70},
]

t0 = time.perf_counter()
best_i = 0
best_d = math.inf
distances_km = []
for i, d in enumerate(drones):
    dist = haversine_km(d["lat"], d["lon"], MISSION_LAT, MISSION_LON)
    distances_km.append(dist)
    if dist < best_d:
        best_d, best_i = dist, i
classical_time_s = time.perf_counter() - t0

classical_name = drones[best_i]["name"]
classical_dist_km = best_d
print("Classical pick:", classical_name)
print(f"Distance to mission: {classical_dist_km:.3f} km")
print(f"Wall time (loop): {classical_time_s * 1e6:.2f} µs")

### What this section does

We encode **“pick exactly one of five drones”** as **five qubits** in a one-hot style: each basis bitstring with a single `1` means “that drone is selected.” Invalid patterns (none or several drones) get a **large penalty** on the diagonal cost so the optimizer discourages them.

The **cost Hamiltonian** rewards short distances by placing **`-distance`** on the diagonal for valid one-hot states (maximizing expected energy ≈ pushing toward **minimum distance**).

We then build a **depth-1 QAOA** circuit: Hadamard start, `exp(-iγ H_C)`, `exp(-iβ H_M)` with `H_M = Σ X_i`. We **classically** search a small grid of `(γ, β)` to maximize `⟨H_C⟩` via `StatevectorEstimator`, bind those angles, and **draw the circuit** for the slide deck. Finally we **time** transpilation + Aer simulation with shots and decode the most likely **valid** bitstring.

*Note:* The heavy classical work here is the **simulator** and parameter search — not a claim of quantum speedup on five drones.

In [ ]:
distances = np.array(distances_km, dtype=float)
invalid_penalty = -500.0
# Higher diagonal energy is better: use -distance so min distance ⟺ max eigenvalue contribution
drone_scores = (-distances).tolist()
num_qubits = len(drones)


def build_diagonal_scores(n, scores, penalty):
    dim = 1 << n
    diag = [0.0] * dim
    for idx in range(dim):
        bits = [(idx >> j) & 1 for j in range(n)]
        if sum(bits) != 1:
            diag[idx] = penalty
            continue
        drone_index = bits.index(1)
        diag[idx] = scores[drone_index]
    return diag


def diagonal_to_sparse_pauli_z(diag):
    """Expand diagonal H in the computational basis as sum of Pauli Z strings."""
    dim = len(diag)
    n = dim.bit_length() - 1
    terms = []
    inv_dim = 1.0 / dim
    for a in range(dim):
        coeff = 0.0
        for x in range(dim):
            parity = (a & x).bit_count() & 1
            coeff += float(diag[x]) * (-1.0 if parity else 1.0)
        coeff *= inv_dim
        if abs(coeff) < 1e-12:
            continue
        label_chars = []
        for qi in range(n - 1, -1, -1):
            label_chars.append("Z" if (a >> qi) & 1 else "I")
        terms.append(("".join(label_chars), coeff))
    return SparsePauliOp.from_list(terms)


def sum_pauli_x_mixer(n):
    ops = [SparsePauliOp(Pauli("".join("X" if i == j else "I" for j in range(n)))) for i in range(n)]
    return sum(ops[1:], ops[0])


def build_qaoa_circuit(cost_h, mixer_h, reps=1):
    n = cost_h.num_qubits
    gammas = ParameterVector("γ", reps)
    betas = ParameterVector("β", reps)
    qc = QuantumCircuit(n)
    qc.h(range(n))
    for layer in range(reps):
        qc.append(PauliEvolutionGate(cost_h, gammas[layer]), range(n))
        qc.append(PauliEvolutionGate(mixer_h, betas[layer]), range(n))
    return qc


diag = build_diagonal_scores(num_qubits, drone_scores, invalid_penalty)
cost_hamiltonian = diagonal_to_sparse_pauli_z(diag)
mixer_hamiltonian = sum_pauli_x_mixer(num_qubits)
qaoa_qc = build_qaoa_circuit(cost_hamiltonian, mixer_hamiltonian, reps=1)

print("QAOA circuit (parameterized; angles set in the next step):")
display(qaoa_qc.draw(output="mpl", fold=80))

# Small grid search for depth-1 QAOA angles
estimator = StatevectorEstimator()
t_q_start = time.perf_counter()
axis_g = np.linspace(0.0, 2 * math.pi, 24, endpoint=False)
axis_b = np.linspace(0.0, math.pi, 20, endpoint=False)
best_params, best_ev = None, -float("inf")
for g in axis_g:
    for b in axis_b:
        params = np.array([g, b], dtype=float)
        job = estimator.run([(qaoa_qc, [cost_hamiltonian], [params])])
        ev = float(job.result()[0].data.evs[0])
        if ev > best_ev:
            best_ev, best_params = ev, params

backend = AerSimulator(method="statevector")
try:
    backend.set_options(max_parallel_threads=1, max_parallel_experiments=1, max_parallel_shots=1)
except Exception:
    pass

measured = qaoa_qc.copy()
measured.measure_all()
bound = measured.assign_parameters(best_params)
compiled = transpile(bound, backend=backend, optimization_level=1)
job = backend.run(compiled, shots=8192)
counts = job.result().get_counts()
quantum_time_s = time.perf_counter() - t_q_start


def one_hot_index_from_bitstring(s):
    if len(s) != num_qubits:
        return None
    bits_lsb_first = [int(ch) for ch in reversed(s)]
    if sum(bits_lsb_first) != 1:
        return None
    return bits_lsb_first.index(1)


best_bs, best_c = None, -1
for bs, c in counts.items():
    idx = one_hot_index_from_bitstring(bs)
    if idx is None:
        continue
    old_idx = one_hot_index_from_bitstring(best_bs) if best_bs is not None else None
    old_score = drone_scores[old_idx] if old_idx is not None else -float("inf")
    if c > best_c or (c == best_c and drone_scores[idx] > old_score):
        best_bs, best_c = bs, c

if best_bs is None:
    # fallback: best one-hot by statevector probability
    sv = Statevector(qaoa_qc.assign_parameters(best_params))
    probs = sv.probabilities()
    best_p = -1.0
    for i in range(len(probs)):
        bits = [(i >> j) & 1 for j in range(num_qubits)]
        if sum(bits) != 1:
            continue
        idx = bits.index(1)
        p = float(probs[i])
        if p > best_p:
            best_p, best_i_sv = p, idx
    quantum_idx = best_i_sv
    quantum_name = drones[quantum_idx]["name"]
else:
    quantum_idx = one_hot_index_from_bitstring(best_bs)
    quantum_name = drones[quantum_idx]["name"]

quantum_dist_km = float(distances[quantum_idx])
print("Optimized ⟨H_C⟩ (best grid):", round(best_ev, 6))
print("Quantum (simulated) pick:", quantum_name)
print(f"Distance to mission: {quantum_dist_km:.3f} km")
print(f"Wall time (grid + Aer shots): {quantum_time_s * 1e3:.2f} ms")

### What this section does

We **summarize outcomes** in a small table: which drone each approach chose, mission distance, and **wall-clock time** for the classical loop vs the **QAOA + simulator** pipeline.

The **bar chart** makes the talking point visual: on five drones the classical method is vastly faster; the quantum line is really showing **workflow cost of simulation**, not hardware advantage. In a talk, this sets up your spoken transition to **regimes where the quantum formulation matters**.

In [ ]:
rows = [
    {
        "Approach": "Classical (argmin distance)",
        "Selected drone": classical_name,
        "Distance (km)": round(classical_dist_km, 4),
        "Time (s)": classical_time_s,
    },
    {
        "Approach": "QAOA + Aer (this notebook)",
        "Selected drone": quantum_name,
        "Distance (km)": round(quantum_dist_km, 4),
        "Time (s)": quantum_time_s,
    },
]

# Pretty table without requiring pandas
print(f"{'Approach':<32} {'Drone':<12} {'Dist km':>10} {'Time (s)':>12}")
print("-" * 70)
for r in rows:
    print(
        f"{r['Approach']:<32} {r['Selected drone']:<12} {r['Distance (km)']:>10.4f} {r['Time (s)']:>12.6f}"
    )

labels = ["Classical\n(argmin)", "QAOA\n(simulator)"]
times_ms = [classical_time_s * 1e3, quantum_time_s * 1e3]
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(labels))
bars = ax.bar(x, times_ms, color=["#2ecc71", "#3498db"], edgecolor="black", linewidth=0.6)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Wall time (ms)")
ax.set_title("Execution time: classical loop vs QAOA + Aer (log scale)")
ax.set_yscale("log")
for b, v in zip(bars, times_ms):
    ax.text(b.get_x() + b.get_width() / 2, v * 1.15, f"{v:.3g} ms", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

match = classical_name == quantum_name
print("\nSame drone selected?", "Yes" if match else "No (shallow QAOA / sampling noise)")

### Takeaway for the audience

For **five** drones, **classical enumeration wins** on time and simplicity — and that is the honest baseline an IEEE crowd expects.

The **quantum value proposition** shows up when the decision problem **scales**: many coupled assets, **combinatorial** routing or assignment layers, and structure that maps naturally to **Ising / QUBO** forms where **heuristics like QAOA** (or hybrid quantum–classical loops) are studied as paths toward better solutions than greedy classical methods alone — especially as **hardware depth and noise** improve and **simulation** is no longer the bottleneck.

This notebook is deliberately small so you can **execute it live** or **export figures** (`circuit.draw`, bar chart) straight into slides.